# Лабораторная работа 4.2. t-тесты

Проверим, отличаются ли средние результаты юношей и девушек по 15 психологическим и учебным показателям.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

alpha = 0.05

df = pd.read_excel('Психологи.xlsx', sheet_name='Лист2')
df.columns = [' '.join(col.split()) for col in df.columns]
df = df.rename(columns={'экстраверсия- интроверсия': 'экстраверсия–интроверсия'})
df['пол'] = df['пол'].str.strip()
boys = df['пол'] == 'м'
girls = df['пол'] == 'ж'
cols = df.drop(columns=['учащиеся', 'пол', 'класс', 'профиль ВУЗа']).columns

print(f'Юношей: {boys.sum()}, девушек: {girls.sum()}. Показателей: {len(cols)}.')
print(f'Пропусков: {df.isna().sum().sum()}.')


Юношей: 23, девушек: 23. Показателей: 15.
Пропусков: 0.


## Гипотезы и выбор теста

Для каждого показателя H₀ означает равенство средних у юношей и девушек, H₁ означает различие средних. Используем двусторонний t-тест для независимых выборок при α = 0,05. Критерием Ливена проверяем равенство дисперсий. При p < 0,05 выбираем Уэлча, иначе используем тест Стьюдента.

Критерий Шапиро–Уилка помогает заметить отклонения от нормальности. Для части балльных показателей такие отклонения есть. При 23 наблюдениях в каждой группе результаты t-теста по этим показателям трактуем как приближённые.

In [2]:
rows = []
for col in cols:
    a = df.loc[boys, col].dropna()
    b = df.loc[girls, col].dropna()
    p_a = stats.shapiro(a).pvalue
    p_b = stats.shapiro(b).pvalue
    p_lev = stats.levene(a, b).pvalue
    equal = p_lev >= alpha
    t, p = stats.ttest_ind(a, b, equal_var=equal, alternative='two-sided')

    var = ((len(a) - 1) * a.var() + (len(b) - 1) * b.var()) / (len(a) + len(b) - 2)
    d = (a.mean() - b.mean()) / np.sqrt(var)
    test = 'Стьюдент' if equal else 'Уэлч'
    decision = 'есть различие' if p < alpha else 'H₀ не отклоняем'
    rows.append([col, a.mean(), b.mean(), p_a, p_b, p_lev, test, t, p, d, decision])

res = pd.DataFrame(rows, columns=[
    'Показатель', 'Среднее Ю', 'Среднее Д', 'p Шапиро Ю', 'p Шапиро Д',
    'p Ливена', 'Тест', 't', 'p', 'd Коэна', 'Вывод',
]).set_index('Показатель')

print('Проверка предпосылок')
print(res[['p Шапиро Ю', 'p Шапиро Д', 'p Ливена', 'Тест']].to_string(
    float_format=lambda v: f'{v:.5g}',
))
print()
print('Сравнение средних')
print(res[['Среднее Ю', 'Среднее Д', 't', 'p', 'd Коэна', 'Вывод']].to_string(
    float_format=lambda v: f'{v:.5g}',
))

Проверка предпосылок
                          p Шапиро Ю  p Шапиро Д  p Ливена      Тест
Показатель                                                          
осведомленность              0.15401     0.45487   0.40198  Стьюдент
скрытые фигуры               0.19846   0.0035624   0.61761  Стьюдент
пропущенные слова            0.55795     0.36228   0.85661  Стьюдент
арифметика                   0.13665      0.7013   0.52702  Стьюдент
понятливость                 0.15693    0.079352   0.88918  Стьюдент
исключение изображений      0.049248   0.0069702   0.16357  Стьюдент
аналогии                     0.42423    0.021982   0.13361  Стьюдент
числовые ряды                0.15994    0.043373   0.51007  Стьюдент
умозаключения               0.069611   0.0085575   0.94584  Стьюдент
геометрическое сложение     0.078436   0.0048298   0.63432  Стьюдент
заучивание слов             0.031263     0.11634   0.65691  Стьюдент
средний IQ                  0.042329    0.099056   0.54532  Стьюдент
экстраверсия–

## Сила различий

Для сравнения показателей с разными шкалами берём модуль d Коэна. Знак d показывает направление разности средних «юноши минус девушки».

In [3]:
rank = res['d Коэна'].abs().sort_values(ascending=False)
print(f'Самое сильное различие: {rank.index[0]} (|d| = {rank.iloc[0]:.3f}).')
print(f'Самое слабое различие: {rank.index[-1]} (|d| = {rank.iloc[-1]:.3f}).')

Самое сильное различие: экстраверсия–интроверсия (|d| = 1.016).
Самое слабое различие: средний IQ (|d| = 0.049).


## Вывод

Статистически значимое различие получено по шкале «экстраверсия–интроверсия». Средний балл у девушек равен 14,61, у юношей 11,57, p = 0,00127. Этот результат сохраняется и с поправкой Бонферрони на 15 сравнений, поскольку 0,00127 < 0,05 / 15.

По остальным показателям оснований отклонить H₀ нет. Ближе всего к порогу результат по скрытым фигурам, p = 0,050442. Самое слабое различие по среднему IQ.